<a href="https://colab.research.google.com/github/nvaprameya-source/AI-Travel-Concierge-AI-Agent-Development-Dual-Track-Version/blob/main/framework_RAG1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install openai gradio faiss-cpu sentence-transformers numpy

In [ ]:
from google.colab import userdata
GROQ_API_KEY=userdata.get('GROQ_API_KEY')

In [ ]:
import json

frameworks = [
    {"type": "itinerary", "content": "For a 3-day trip: Day 1 arrival and light exploration, Day 2 full exploration, Day 3 shopping and departure"},
    {"type": "itinerary", "content": "For short trips: prioritize nearby attractions and minimize travel time"},
    {"type": "budget", "content": "Split budget: 40% stay, 30% food, 20% transport, 10% activities"},
    {"type": "budget", "content": "For low budget travel: use hostels, public transport, and free attractions"},
    {"type": "group_travel", "content": "For group trips: use shared stays and cost splitting"},
    {"type": "solo_travel", "content": "For solo trips: prioritize safety and flexibility"},
    {"type": "activity_mapping", "content": "Adventure: trekking, camping. Relaxation: cafes, sightseeing. Cultural: temples, markets"},
    {"type": "season", "content": "In summer: prefer morning/evening activities, avoid midday heat"}
]

with open("frameworks.json", "w") as f:
    json.dump(frameworks, f)

In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Load model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Load frameworks
import json
with open("frameworks.json", "r") as f:
    data = json.load(f)

texts = [item["content"] for item in data]

# Create embeddings
embeddings = model.encode(texts)

# Create FAISS index
dimension = len(embeddings[0])
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

def retrieve_frameworks(query, k=3):
    query_embedding = model.encode([query])
    distances, indices = index.search(np.array(query_embedding), k)
    return "\n".join([texts[i] for i in indices[0]])

In [ ]:
!pip install langchain_groq

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=GROQ_API_KEY,
    temperature=0.3,
    max_tokens=500
)

In [ ]:
conversation_history = []

SYSTEM_PROMPT = """
You are an AI travel planner.

Use the provided frameworks to generate structured, practical, and realistic travel plans.

Rules:
- Always give a day-wise itinerary
- Include budget breakdown
- Include useful tips
- Avoid hallucinating unknown place names
"""


def agent_loop(user_input):

    # 🔍 Step 1: Retrieve frameworks (RAG)
    context = retrieve_frameworks(user_input)

    # 🧠 Step 2: Build messages
    messages = []

    messages.append({
        "role": "system",
        "content": SYSTEM_PROMPT
    })

    messages.append({
        "role": "system",
        "content": f"Relevant frameworks:\n{context}"
    })

    messages.extend(conversation_history)

    messages.append({
        "role": "user",
        "content": user_input
    })

    # 🔁 Convert messages → prompt (Groq format)
    prompt = ""
    for msg in messages:
        prompt += f"{msg['role'].upper()}: {msg['content']}\n"

    # 🧠 Step 3: Single LLM call
    response = llm.invoke(prompt)

    reply = response.content

    # 💾 Step 4: Save memory
    conversation_history.append({
        "role": "user",
        "content": user_input
    })

    conversation_history.append({
        "role": "assistant",
        "content": reply
    })

    return reply

In [ ]:
print(agent_loop("Plan a 3-day trip to Goa under 8000 with friends"))